# Institutional Quant Research Engine V2

**Purpose:** a single, disciplined research engine for point-in-time features, walk-forward experiments, robustness checks, information-source ablation, portfolio construction, risk controls, and reproducible experiment tracking.

### What changed from the stitched research notebook
- One master pipeline; no duplicated Notebook-3 / Notebook-4 research tracks.
- Explicit **train / validation / untouched test** partitions.
- Point-in-time event joins use **availability time**, not calendar dates.
- Validation-only parameter selection; the locked test is never used for tuning.
- Mandatory robustness diagnostics: costs, slippage, delay, parameter perturbation, bootstrap, regime, placebo, and cross-asset where data permits.
- Every run receives an immutable experiment ID and a leaderboard record.
- Research results are separated from paper/live execution.

> **Default mode is synthetic/offline** so the notebook can execute end-to-end without network credentials. Change `DATA_MODE` when ready for live historical data.

## Research contract

A strategy is **not promoted because it has the highest backtest Sharpe**. Promotion requires clean OOS evaluation, economic plausibility, cost/slippage survival, stability, and an explicit trial count. The locked test period is evaluation-only.

In [1]:
from __future__ import annotations
import os, json, math, hashlib, warnings, platform, time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional, Iterable

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

DATA_MODE = "synthetic"   # "synthetic", "csv", or "yfinance"
ASSETS = ["SPY", "QQQ", "AAPL", "MSFT", "JPM", "GLD", "TLT"]
TARGET = "SPY"
START = "2012-01-01"
END = "2026-01-01"
BAR_FREQ = "1D"

COST_BPS = 5.0
SLIPPAGE_BPS = 1.0
TARGET_VOL = 0.10
MAX_POSITION = 1.0

ARTIFACT_DIR = Path("/mnt/data/research_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)
CACHE_DIR = Path("/mnt/data/data_cache_v2")
CACHE_DIR.mkdir(exist_ok=True)

print(f"SEED={SEED} | DATA_MODE={DATA_MODE} | TARGET={TARGET}")

SEED=42 | DATA_MODE=synthetic | TARGET=SPY


## 1. Data layer

Raw observations and alternative-data events are kept separate from processed features. Every observation/event carries timestamps that let the feature engine enforce information availability.

In [2]:
def _stable_hash(obj) -> str:
    raw = json.dumps(obj, sort_keys=True, default=str).encode()
    return hashlib.sha256(raw).hexdigest()[:16]


def generate_synthetic_ohlcv(assets, start=START, end=END, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = pd.bdate_range(start, end, inclusive="left")
    prices = {}
    volumes = {}
    common = rng.normal(0, 0.0004, len(idx))
    drifts = np.linspace(0.0001, 0.0004, len(assets))
    for i, asset in enumerate(assets):
        eps = rng.normal(0, 0.010 + 0.001*i, len(idx))
        latent = common + eps + drifts[i]
        # deterministic-ish crisis/regime component for meaningful stress tests
        shock = np.zeros(len(idx))
        for a, b, mag in [("2020-02-01","2020-04-01",-0.0025),("2022-01-01","2022-10-01",-0.0010),("2024-01-01","2024-04-01",0.0007)]:
            m=(idx>=a)&(idx<b); shock[m]=mag
        ret = latent + shock
        close = 100*np.exp(np.cumsum(ret))
        base_vol = 1_000_000 * (1 + 0.25*np.abs(ret)/max(np.std(ret),1e-8))
        vol = base_vol * rng.lognormal(0,0.20,len(idx))
        prices[asset]=close; volumes[asset]=vol
    return pd.DataFrame(prices,index=idx), pd.DataFrame(volumes,index=idx)


def load_ohlcv(mode=DATA_MODE, assets=ASSETS, start=START, end=END, csv_path=None):
    if mode == "synthetic":
        return generate_synthetic_ohlcv(assets,start,end)
    if mode == "csv":
        if not csv_path:
            raise ValueError("csv_path is required when DATA_MODE='csv'")
        raw = pd.read_csv(csv_path, parse_dates=[0])
        raw = raw.rename(columns={raw.columns[0]:"timestamp"}).set_index("timestamp").sort_index()
        close_cols=[c for c in raw.columns if c.startswith("Close_")]
        vol_cols=[c for c in raw.columns if c.startswith("Volume_")]
        if not close_cols:
            raise ValueError("CSV must contain Close_<ASSET> columns")
        close=raw[close_cols].rename(columns=lambda c:c.replace("Close_",""))
        volume=raw[vol_cols].rename(columns=lambda c:c.replace("Volume_","")) if vol_cols else pd.DataFrame(index=raw.index)
        return close, volume
    if mode == "yfinance":
        import yfinance as yf
        tickers=sorted(set(assets))
        raw=yf.download(tickers, start=start, end=end, interval="1d", auto_adjust=True, progress=False, group_by="column")
        if raw is None or raw.empty:
            raise RuntimeError("No data returned by yfinance")
        if isinstance(raw.columns, pd.MultiIndex):
            close=raw["Close"].copy(); volume=raw["Volume"].copy()
        else:
            t=tickers[0]; close=pd.DataFrame({t:raw["Close"]}, index=raw.index); volume=pd.DataFrame({t:raw["Volume"]}, index=raw.index)
        close=close.sort_index(); volume=volume.reindex(close.index)
        return close, volume
    raise ValueError(f"Unknown DATA_MODE={mode}")

def save_raw_snapshot(close_df: pd.DataFrame, volume_df: pd.DataFrame, label: str = "raw"):
    stamp=time.strftime("%Y%m%dT%H%M%SZ",time.gmtime())
    path=ARTIFACT_DIR/f"{label}_{stamp}.parquet"
    raw=pd.concat({"Close":close_df,"Volume":volume_df},axis=1)
    try:
        raw.to_parquet(path)
    except Exception:
        path=path.with_suffix(".csv")
        raw.to_csv(path)
    return path

close, volume = load_ohlcv()
raw_snapshot_path=save_raw_snapshot(close,volume)
print("raw snapshot:",raw_snapshot_path)
close = close.sort_index()
volume = volume.reindex(close.index)
print("close:", close.shape, "date range:", close.index.min().date(), "->", close.index.max().date())

raw snapshot: /mnt/data/research_artifacts/raw_20260906T205932Z.csv
close: (3653, 7) date range: 2012-01-02 -> 2025-12-31


## 2. Point-in-time event schema and guardrails

The core rule is: **a feature may only consume an event after `availability_time`.** `publication_time` is retained separately because it is not always the same as market availability.

In [3]:
EVENT_COLUMNS = [
    "event_id","symbol","event_time","publication_time","availability_time",
    "source","raw_value","processed_value","sentiment","engagement","text"
]


def normalize_events(events: pd.DataFrame) -> pd.DataFrame:
    x = events.copy()
    defaults = {
        "event_id": "", "symbol": "", "event_time": pd.NaT, "publication_time": pd.NaT,
        "availability_time": pd.NaT, "source": "other", "raw_value": np.nan,
        "processed_value": np.nan, "sentiment": 0.0, "engagement": 0.0, "text": ""
    }
    for c,v in defaults.items():
        if c not in x: x[c]=v
    for c in ["event_time","publication_time","availability_time"]:
        x[c]=pd.to_datetime(x[c], utc=True, errors="coerce")
    x["symbol"]=x["symbol"].astype(str).str.upper().str.strip()
    x["source"]=x["source"].astype(str).str.lower().str.strip()
    x["sentiment"]=pd.to_numeric(x["sentiment"], errors="coerce").fillna(0).clip(-1,1)
    x["engagement"]=pd.to_numeric(x["engagement"], errors="coerce").fillna(0).clip(lower=0)
    x["text"]=x["text"].fillna("").astype(str)
    x["event_id"]=x["event_id"].where(x["event_id"].ne(""), [
        _stable_hash([a,b,c,d]) for a,b,c,d in zip(x["symbol"],x["event_time"],x["source"],x["text"].str[:160])
    ])
    # Never silently treat missing availability as immediately tradable.
    x["availability_time"]=x["availability_time"].fillna(x["publication_time"])
    return x.sort_values(["availability_time","event_id"]).drop_duplicates("event_id").reset_index(drop=True)


def information_guardrails(events, bars_index):
    ev=normalize_events(events)
    violations=[]
    if ev["availability_time"].isna().any():
        violations.append(f"{int(ev['availability_time'].isna().sum())} events have no availability/publication time")
    if (ev["availability_time"] < ev["event_time"]).fillna(False).any():
        violations.append("availability_time precedes event_time for at least one event")
    b=pd.DatetimeIndex(bars_index)
    return {
        "events": int(len(ev)),
        "sources": sorted(ev["source"].unique().tolist()),
        "violations": violations,
        "status": "PASS" if not violations else "REVIEW"
    }

empty_events = pd.DataFrame(columns=EVENT_COLUMNS)
print(information_guardrails(empty_events, close.index))

{'events': 0, 'sources': [], 'violations': [], 'status': 'PASS'}


## 3. Feature engine

Features are computed from information known at or before the bar timestamp. Cross-sectional features use only contemporaneous assets. No backfill is used.

In [4]:
def build_price_features(close_df: pd.DataFrame, volume_df: pd.DataFrame) -> pd.DataFrame:
    feat={}
    for asset in close_df.columns:
        px=close_df[asset].astype(float)
        ret1=px.pct_change()
        for lag in [1,3,5,10,20,60]: feat[f"ret_{lag}_{asset}"]=px.pct_change(lag)
        feat[f"vol_20_{asset}"]=ret1.rolling(20).std()
        feat[f"vol_60_{asset}"]=ret1.rolling(60).std()
        feat[f"trend_20_60_{asset}"]=px.rolling(20).mean()/px.rolling(60).mean()-1
        feat[f"dist_sma20_{asset}"]=px/px.rolling(20).mean()-1
        delta=px.diff(); gain=delta.clip(lower=0).rolling(14).mean(); loss=(-delta.clip(upper=0)).rolling(14).mean()
        rs=gain/(loss+1e-12); feat[f"rsi14_{asset}"]=100-(100/(1+rs))
        if asset in volume_df:
            v=volume_df[asset].astype(float); vm=v.rolling(20).mean(); vs=v.rolling(20).std()
            feat[f"volume_z20_{asset}"]=(v-vm)/(vs+1e-12)
    if TARGET in close_df:
        tr=close_df[TARGET].pct_change()
        feat["target_vol20"]=tr.rolling(20).std()
        feat["target_vol60"]=tr.rolling(60).std()
        for asset in close_df.columns:
            if asset!=TARGET:
                feat[f"corr_60_{asset}"]=tr.rolling(60).corr(close_df[asset].pct_change())
    f=pd.DataFrame(feat,index=close_df.index)
    # market regime is based only on current/past values
    if TARGET in close_df:
        sma200=close_df[TARGET].rolling(200).mean()
        f["bull_regime"]=(close_df[TARGET] > sma200).astype(float)
    return f

features=build_price_features(close,volume)
print("features:",features.shape)

features: (3653, 93)


In [5]:
def add_point_in_time_information(features: pd.DataFrame, events: pd.DataFrame, bars_index) -> pd.DataFrame:
    ev=normalize_events(events)
    out=features.copy()
    for c in ["info_count","info_attention","info_sentiment","info_abs_sentiment","info_source_count","info_disagreement","info_novelty"]:
        out[c]=0.0
    if ev.empty: return out
    idx_orig=pd.DatetimeIndex(bars_index)
    idx_utc=idx_orig.tz_localize("UTC") if idx_orig.tz is None else idx_orig
    # Efficient per-symbol as-of accumulation; no row after the bar is visible.
    for symbol, g in ev.groupby("symbol"):
        if symbol != TARGET: continue
        g=g.sort_values("availability_time").copy()
        for label, ts in zip(idx_orig, idx_utc):
            prior=g[g["availability_time"]<=ts]
            if prior.empty: continue
            age_h=(ts-prior["availability_time"]).dt.total_seconds()/3600
            prior=prior[(age_h>=0)&(age_h<=72)]
            if prior.empty: continue
            w=(np.exp(-age_h.loc[prior.index]/24.0)*prior["engagement"].pow(0.25).replace(0,1)).to_numpy(float)
            s=prior["sentiment"].to_numpy(float)
            total=max(w.sum(),1e-12)
            out.at[label,"info_count"]=len(prior)
            out.at[label,"info_attention"]=float(w.sum())
            out.at[label,"info_sentiment"]=float(np.dot(w,s)/total)
            out.at[label,"info_abs_sentiment"]=float(np.dot(w,np.abs(s))/total)
            out.at[label,"info_source_count"]=float(prior["source"].nunique())
            per=prior.groupby("source")["sentiment"].mean()
            out.at[label,"info_disagreement"]=float(per.std(ddof=0)) if len(per)>1 else 0.0
            out.at[label,"info_novelty"]=float(1-prior["event_id"].duplicated().mean())
    return out

# Optional sample event table; real providers should populate these fields.
sample_events=pd.DataFrame([
    {"event_id":"e1","symbol":TARGET,"event_time":"2023-01-03T12:00:00Z","publication_time":"2023-01-03T12:01:00Z","availability_time":"2023-01-03T12:02:00Z","source":"news","sentiment":0.8,"engagement":20,"text":"positive sample"},
    {"event_id":"e2","symbol":TARGET,"event_time":"2023-01-03T15:00:00Z","publication_time":"2023-01-03T15:01:00Z","availability_time":"2023-01-03T15:02:00Z","source":"news","sentiment":-0.4,"engagement":50,"text":"negative sample"},
])
features_with_info=add_point_in_time_information(features,sample_events,close.index)
print(information_guardrails(sample_events,close.index))

{'events': 2, 'sources': ['news'], 'violations': [], 'status': 'PASS'}


## 4. Labels and strict chronological splits

The default target is next-bar direction. The split utility supports a purge gap so overlapping observations cannot leak across train/validation/test boundaries.

In [6]:
def make_labels(close_df, target=TARGET):
    fwd=close_df[target].pct_change().shift(-1)
    y=(fwd>0).astype(float)
    y[fwd.isna()]=np.nan
    return y, fwd

y, fwd_returns = make_labels(close)

@dataclass(frozen=True)
class SplitConfig:
    train_years: int = 5
    validation_years: int = 1
    test_years: int = 1
    purge_days: int = 5
    step_years: int = 1


def walk_forward_splits(index, cfg=SplitConfig()):
    idx=pd.DatetimeIndex(index).sort_values()
    start=idx.min().normalize(); end=idx.max().normalize()
    cur=start + pd.DateOffset(years=cfg.train_years)
    while cur + pd.DateOffset(years=cfg.validation_years+cfg.test_years) <= end:
        train_start=cur-pd.DateOffset(years=cfg.train_years)
        train_end=cur
        val_end=cur+pd.DateOffset(years=cfg.validation_years)
        test_end=val_end+pd.DateOffset(years=cfg.test_years)
        purge=pd.Timedelta(days=cfg.purge_days)
        tr=idx[(idx>=train_start)&(idx<train_end-purge)]
        va=idx[(idx>=train_end+purge)&(idx<val_end-purge)]
        te=idx[(idx>=val_end+purge)&(idx<test_end)]
        if len(tr) and len(va) and len(te): yield tr,va,te
        cur=cur+pd.DateOffset(years=cfg.step_years)

splits=list(walk_forward_splits(features.index,SplitConfig()))
print("walk-forward folds:",len(splits))

walk-forward folds: 7


## 5. Model and trading engine

A deliberately compact baseline is used here. The important improvement is **evaluation discipline**, not a larger model. Parameters are fit on train, selected on validation, and only then scored on the fold's test period.

In [7]:
BASE_FEATURES=[c for c in features_with_info.columns if c in features_with_info.columns and not c.startswith("future_")]


def fit_model(X, y):
    pipe=Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler()),
        ("model",LogisticRegression(C=0.5, max_iter=2000, random_state=SEED))
    ])
    pipe.fit(X,y)
    return pipe


def max_drawdown(cum):
    cum=pd.Series(cum,dtype=float)
    return float((cum/cum.cummax()-1).min())


def sharpe(rets, annualization=252):
    r=pd.Series(rets,dtype=float).dropna()
    if len(r)<2 or r.std(ddof=1)==0: return np.nan
    return float(np.sqrt(annualization)*r.mean()/r.std(ddof=1))


def sortino(rets, annualization=252):
    r=pd.Series(rets,dtype=float).dropna(); downside=r[r<0].std(ddof=1)
    return float(np.sqrt(annualization)*r.mean()/downside) if len(r)>1 and downside>0 else np.nan


def backtest_from_probabilities(probs, fwd, threshold=0.55, fee_bps=COST_BPS, slippage_bps=SLIPPAGE_BPS, target_vol=TARGET_VOL, max_position=MAX_POSITION):
    f=pd.Series(fwd,dtype=float)
    p=pd.Series(probs,index=f.index,dtype=float)
    hist_ret=f.shift(1)
    vol=hist_ret.rolling(20).std()
    raw_pos=(p>=threshold).astype(float)
    scale=(target_vol/(vol*np.sqrt(252)+1e-12)).clip(upper=max_position).fillna(0.0)
    pos=raw_pos*scale
    turnover=pos.diff().abs().fillna(pos.abs())
    costs=turnover*(fee_bps+slippage_bps)/10000.0
    strat=pos*f-costs
    cum=(1+strat.fillna(0)).cumprod()
    return {
        "returns":strat,
        "equity":cum,
        "position":pos,
        "turnover":turnover,
        "sharpe":sharpe(strat),
        "sortino":sortino(strat),
        "cagr":float(cum.iloc[-1]**(252/max(len(cum),1))-1) if len(cum) else np.nan,
        "max_dd":max_drawdown(cum),
        "trades":int((turnover>1e-9).sum()),
    }


def choose_threshold(Xv,yv,fwdv, candidates=np.arange(0.50,0.76,0.02), fee_bps=COST_BPS, slippage_bps=SLIPPAGE_BPS):
    # Threshold is chosen only on validation data.
    rows=[]
    for t in candidates:
        # placeholder model is fitted outside so caller supplies Xv probabilities
        pass
    raise RuntimeError("Internal API error: use choose_threshold_from_probs")


def choose_threshold_from_probs(probs, fwdv, candidates=np.arange(0.50,0.76,0.02)):
    rows=[]
    for t in candidates:
        bt=backtest_from_probabilities(probs,fwdv,threshold=float(t))
        rows.append((float(t),bt["sharpe"],bt["max_dd"],bt["trades"]))
    tab=pd.DataFrame(rows,columns=["threshold","sharpe","max_dd","trades"])
    tab=tab.replace([np.inf,-np.inf],np.nan).dropna(subset=["sharpe"])
    if tab.empty:
        return 0.55, pd.DataFrame(rows,columns=["threshold","sharpe","max_dd","trades"])
    return float(tab.sort_values(["sharpe","max_dd"],ascending=[False,False]).iloc[0]["threshold"]), tab

## 6. Experiment runner

Each fold writes a self-contained record. The fold's test set is never used to choose features, hyperparameters, thresholds, calibration, or risk settings.

In [8]:
def run_experiment(features_df, y, fwd, split_cfg=SplitConfig(), use_info=True, feature_subset=None):
    X=features_df.copy()
    common=X.index.intersection(y.dropna().index).intersection(fwd.dropna().index)
    X=X.loc[common]; yy=y.loc[common].astype(int); ff=fwd.loc[common]
    if feature_subset is None: feature_subset=list(X.columns)
    fold_rows=[]; preds=[]
    for fold_id,(tr_idx,va_idx,te_idx) in enumerate(walk_forward_splits(X.index,split_cfg),1):
        tr=tr_idx.intersection(X.index); va=va_idx.intersection(X.index); te=te_idx.intersection(X.index)
        model=fit_model(X.loc[tr,feature_subset],yy.loc[tr])
        pv=model.predict_proba(X.loc[va,feature_subset])[:,1]
        tv,threshold_table=choose_threshold_from_probs(pv,ff.loc[va])
        pt=model.predict_proba(X.loc[te,feature_subset])[:,1]
        bt=backtest_from_probabilities(pt,ff.loc[te],threshold=tv)
        auc=float(roc_auc_score(yy.loc[te],pt)) if yy.loc[te].nunique()>1 else np.nan
        brier=float(brier_score_loss(yy.loc[te],pt))
        fold_rows.append({
            "fold":fold_id,"train_start":str(tr.min().date()),"train_end":str(tr.max().date()),
            "val_start":str(va.min().date()),"val_end":str(va.max().date()),
            "test_start":str(te.min().date()),"test_end":str(te.max().date()),
            "threshold":tv,"oos_sharpe":bt["sharpe"],"oos_sortino":bt["sortino"],"oos_cagr":bt["cagr"],
            "oos_max_dd":bt["max_dd"],"oos_trades":bt["trades"],"oos_turnover":float(bt["turnover"].sum()),
            "oos_auc":auc,"oos_brier":brier,
        })
        preds.append(pd.DataFrame({"prob":pt,"y":yy.loc[te].values,"fwd":ff.loc[te].values,"fold":fold_id},index=te))
    folds=pd.DataFrame(fold_rows)
    pred_df=pd.concat(preds).sort_index() if preds else pd.DataFrame()
    return folds,pred_df

folds,pred_df=run_experiment(features_with_info,y,fwd_returns)
print(folds)

   fold train_start   train_end  ... oos_turnover   oos_auc oos_brier
0     1  2012-01-02  2016-12-27  ...    40.042164  0.474994  0.289916
1     2  2013-01-02  2017-12-27  ...    41.017592  0.537370  0.263647
2     3  2014-01-02  2018-12-27  ...    38.287169  0.506954  0.281506
3     4  2015-01-02  2019-12-27  ...    23.598541  0.523168  0.273447
4     5  2016-01-04  2020-12-25  ...    18.674415  0.471123  0.315273
5     6  2017-01-02  2021-12-27  ...    24.606656  0.546226  0.275275
6     7  2018-01-02  2022-12-27  ...    12.127973  0.452869  0.356014

[7 rows x 16 columns]


## 7. Robustness / anti-overfitting battery

These tests operate on the **same untouched OOS predictions** where possible. No test result is fed back into model fitting.

In [9]:
def cost_sensitivity(pred_df, fee_grid=(0,2.5,5,10,20), slippage_bps=SLIPPAGE_BPS, threshold_by_fold=None):
    rows=[]
    for fee in fee_grid:
        all_r=[]
        for fold, g in pred_df.groupby("fold"):
            threshold=float(threshold_by_fold.get(int(fold),0.55)) if threshold_by_fold else 0.55
            bt=backtest_from_probabilities(g["prob"],g["fwd"],threshold=threshold,fee_bps=fee,slippage_bps=slippage_bps)
            all_r.append(bt["returns"])
        r=pd.concat(all_r).sort_index()
        rows.append({"fee_bps":fee,"sharpe":sharpe(r),"max_dd":max_drawdown((1+r).cumprod()),"turnover":np.nan})
    return pd.DataFrame(rows)


def delay_test(pred_df, delays=(0,1,2,3), threshold=0.55):
    rows=[]
    for d in delays:
        all_r=[]
        for _,g in pred_df.groupby("fold"):
            p=g["prob"].shift(d).fillna(0.5)
            bt=backtest_from_probabilities(p,g["fwd"],threshold=threshold)
            all_r.append(bt["returns"])
        r=pd.concat(all_r).sort_index()
        rows.append({"delay_bars":d,"sharpe":sharpe(r),"max_dd":max_drawdown((1+r).cumprod())})
    return pd.DataFrame(rows)


def bootstrap_sharpe_ci(returns, n=2000, seed=SEED):
    r=pd.Series(returns).dropna().to_numpy()
    if len(r)<20: return {"lo":np.nan,"median":np.nan,"hi":np.nan}
    rng=np.random.default_rng(seed)
    vals=[]
    for _ in range(n): vals.append(sharpe(pd.Series(rng.choice(r,size=len(r),replace=True))))
    return {"lo":float(np.nanpercentile(vals,2.5)),"median":float(np.nanpercentile(vals,50)),"hi":float(np.nanpercentile(vals,97.5))}


def parameter_perturbation(feature_df, y, fwd, configs):
    rows=[]
    for i,cfg in enumerate(configs,1):
        fs, sc = cfg
        folds_i,_=run_experiment(feature_df,y,fwd,split_cfg=sc,feature_subset=fs)
        rows.append({"variant":i,"median_oos_sharpe":float(folds_i["oos_sharpe"].median()),"mean_oos_sharpe":float(folds_i["oos_sharpe"].mean()),"positive_folds":int((folds_i["oos_sharpe"]>0).sum()),"n_folds":len(folds_i)})
    return pd.DataFrame(rows)

cost_table=cost_sensitivity(pred_df)
delay_table=delay_test(pred_df)
combined_rets=pd.concat([backtest_from_probabilities(g["prob"],g["fwd"],threshold=0.55)["returns"] for _,g in pred_df.groupby("fold")]).sort_index()
boot=bootstrap_sharpe_ci(combined_rets)
print("Cost sensitivity:", cost_table)
print("Delay sensitivity:", delay_table)
print("Bootstrap Sharpe CI:",boot)

Cost sensitivity:    fee_bps    sharpe    max_dd  turnover
0      0.0  0.181247 -0.140305       NaN
1      2.5  0.083507 -0.141104       NaN
2      5.0 -0.014306 -0.154923       NaN
3     10.0 -0.209939 -0.202190       NaN
4     20.0 -0.599770 -0.311079       NaN
Delay sensitivity:    delay_bars    sharpe    max_dd
0           0 -0.014306 -0.154923
1           1 -0.031037 -0.149425
2           2 -0.058622 -0.143036
3           3  0.480355 -0.123514
Bootstrap Sharpe CI: {'lo': -0.7901314478556348, 'median': -0.015920599791841246, 'hi': 0.7048613916766397}


## 8. Parameter perturbation

Nearby split configurations are re-run without touching the main OOS outputs. Robustness is a property of a neighborhood, not one tuned point.

In [10]:
perturbations=parameter_perturbation(
    features_with_info,y,fwd_returns,
    configs=[
        (BASE_FEATURES, SplitConfig(train_years=4, validation_years=1, test_years=1, purge_days=5)),
        (BASE_FEATURES, SplitConfig(train_years=5, validation_years=1, test_years=1, purge_days=5)),
        (BASE_FEATURES, SplitConfig(train_years=6, validation_years=1, test_years=1, purge_days=5)),
    ],
)
print(perturbations)

   variant  median_oos_sharpe  mean_oos_sharpe  positive_folds  n_folds
0        1           0.065913          0.10945               5        8
1        2           0.408517          0.20729               5        7
2        3          -0.042865         -0.35954               3        6


## 8. Placebo / randomized-feature test

A placebo feature should not create a durable OOS effect. This is a sanity check against accidental leakage and search artifacts.

In [11]:
def placebo_test(feature_df, y, fwd, n=5):
    rng=np.random.default_rng(SEED)
    rows=[]
    for i in range(n):
        X=feature_df.copy()
        c=f"__placebo_{i}"
        X[c]=rng.permutation(X.index.size)
        folds,_=run_experiment(X,y,fwd,feature_subset=[c])
        rows.append({"placebo":i,"median_sharpe":float(folds.oos_sharpe.median()),"mean_sharpe":float(folds.oos_sharpe.mean())})
    return pd.DataFrame(rows)

placebo=placebo_test(features_with_info,y,fwd_returns,n=5)
print(placebo)

   placebo  median_sharpe  mean_sharpe
0        0       0.854183     0.577553
1        1       0.629641     0.274235
2        2       0.860108     0.577325
3        3       0.595172     0.495702
4        4      -0.003093    -0.058319


## 9. Information-source ablation

The source-by-source question is whether alternative information adds **incremental OOS value** after price/volume features. The default sample has two news events only to exercise the data path; real source data should be loaded before interpreting this table.

In [12]:
def information_ablation(features_base, features_info, y, fwd):
    price_cols=[c for c in features_base.columns]
    info_cols=[c for c in features_info.columns if c.startswith("info_")]
    rows=[]
    for name, cols in [("price_volume",price_cols),("price_plus_information",price_cols+info_cols)]:
        folds,_=run_experiment(features_info[cols],y,fwd,feature_subset=cols)
        rows.append({
            "model":name,
            "mean_oos_sharpe":float(folds.oos_sharpe.mean()),
            "median_oos_sharpe":float(folds.oos_sharpe.median()),
            "mean_auc":float(folds.oos_auc.mean()),
            "mean_brier":float(folds.oos_brier.mean()),
            "positive_folds":int((folds.oos_sharpe>0).sum()),
            "folds":int(len(folds)),
        })
    return pd.DataFrame(rows)

ablation=information_ablation(features,features_with_info,y,fwd_returns)
print(ablation)

                    model  mean_oos_sharpe  ...  positive_folds  folds
0            price_volume          0.20729  ...               5      7
1  price_plus_information          0.20729  ...               5      7

[2 rows x 7 columns]


## 10. Regime and cross-asset diagnostics

In [13]:
def regime_report(pred_df, close_df, threshold=0.55):
    rows=[]
    for regime_name, mask in {
        "all": pd.Series(True,index=close_df.loc[pred_df.index].index),
        "bull": close_df[TARGET].loc[pred_df.index] > close_df[TARGET].rolling(200).mean().loc[pred_df.index],
        "bear": close_df[TARGET].loc[pred_df.index] <= close_df[TARGET].rolling(200).mean().loc[pred_df.index],
        "high_vol": close_df[TARGET].pct_change().rolling(20).std().loc[pred_df.index] > close_df[TARGET].pct_change().rolling(20).std().rolling(252).median().loc[pred_df.index],
        "low_vol": close_df[TARGET].pct_change().rolling(20).std().loc[pred_df.index] <= close_df[TARGET].pct_change().rolling(20).std().rolling(252).median().loc[pred_df.index],
    }.items():
        g=pred_df.loc[mask.reindex(pred_df.index).fillna(False)]
        if g.empty: continue
        bt=backtest_from_probabilities(g["prob"],g["fwd"],threshold=threshold)
        rows.append({"regime":regime_name,"sharpe":bt["sharpe"],"max_dd":bt["max_dd"],"trades":bt["trades"],"n":len(g)})
    return pd.DataFrame(rows)

regimes=regime_report(pred_df,close)
print(regimes)

     regime    sharpe    max_dd  trades     n
0       all  0.066340 -0.152262    1030  1801
1      bull  0.042328 -0.141685     637  1157
2      bear  0.166755 -0.083168     394   644
3  high_vol  0.358800 -0.112264     530   888
4   low_vol -0.179363 -0.140231     521   913


## 11. Portfolio construction and risk engine

These are deliberately conservative building blocks. They sit **after** strategy-level OOS validation rather than masking a weak signal.

In [14]:
def inverse_vol_weights(vols, cap=0.40):
    v=pd.Series(vols,dtype=float).replace([np.inf,-np.inf],np.nan).dropna()
    inv=1/(v+1e-12)
    w=inv/inv.sum()
    w=w.clip(upper=cap)
    return w/w.sum()


def portfolio_risk(returns_df, weights):
    r=returns_df.dropna(how="all").fillna(0)
    w=pd.Series(weights).reindex(r.columns).fillna(0)
    p=r.dot(w)
    eq=(1+p).cumprod()
    return {
        "annualized_vol":float(p.std(ddof=1)*np.sqrt(252)),
        "sharpe":sharpe(p),
        "max_dd":max_drawdown(eq),
        "beta_to_target":float(p.cov(r[TARGET])/r[TARGET].var()) if TARGET in r and r[TARGET].var()>0 else np.nan,
        "concentration_hhi":float((w[w>0]**2).sum()),
    }

asset_returns=close.pct_change().dropna()
ann_vol=asset_returns.rolling(60).std().iloc[-1]*np.sqrt(252)
weights=inverse_vol_weights(ann_vol)
portfolio_stats=portfolio_risk(asset_returns,weights)
print("weights:", weights)
print("risk:",portfolio_stats)

weights: SPY     0.187390
QQQ     0.158838
AAPL    0.156712
MSFT    0.132167
JPM     0.120117
GLD     0.113935
TLT     0.130840
Name: 2025-12-31 00:00:00, dtype: float64
risk: {'annualized_vol': 0.07695503504565537, 'sharpe': 0.8038081358995832, 'max_dd': -0.22720399207713882, 'beta_to_target': 0.18753491866766622, 'concentration_hhi': 0.146899990790214}


## 12. Standardized experiment registry / leaderboard

Each run records dataset, feature version, strategy version, fold count, trial count, OOS metrics, costs, and robustness evidence. The registry is append-only.

In [15]:
@dataclass
class ExperimentRecord:
    experiment_id: str
    timestamp_utc: str
    strategy: str
    target: str
    universe: list
    data_mode: str
    seed: int
    feature_version: str
    strategy_version: str
    n_trials: int
    n_folds: int
    mean_oos_sharpe: float
    median_oos_sharpe: float
    mean_oos_auc: float
    mean_oos_brier: float
    worst_oos_dd: float
    total_oos_trades: int
    bootstrap_lo: float
    bootstrap_hi: float
    cost_5bps_sharpe: float
    delay_1bar_sharpe: float
    placebo_mean_sharpe: float
    information_incremental_sharpe: float
    pass_fail: str


def create_experiment_record(folds, pred_df, cost_table, delay_table, placebo, ablation):
    cost5=float(cost_table.loc[np.isclose(cost_table.fee_bps,5.0),"sharpe"].iloc[0]) if any(np.isclose(cost_table.fee_bps,5.0)) else np.nan
    d1=float(delay_table.loc[delay_table.delay_bars==1,"sharpe"].iloc[0]) if (delay_table.delay_bars==1).any() else np.nan
    info_delta=np.nan
    if len(ablation)>=2: info_delta=float(ablation.loc[ablation.model.eq("price_plus_information"),"mean_oos_sharpe"].iloc[0]-ablation.loc[ablation.model.eq("price_volume"),"mean_oos_sharpe"].iloc[0])
    checks=[
        float(folds.oos_sharpe.median())>0,
        cost5>0,
        d1>0,
        float(placebo.mean_sharpe.mean())<float(folds.oos_sharpe.median()),
        abs(float(folds.worst_oos_dd.iloc[0] if False else folds.oos_max_dd.min()))<0.50
    ]
    status="PASS" if sum(checks)>=4 else "REVIEW"
    cfg={"ASSETS":ASSETS,"TARGET":TARGET,"DATA_MODE":DATA_MODE,"seed":SEED,"cost_bps":COST_BPS,"slippage_bps":SLIPPAGE_BPS}
    eid=time.strftime("%Y%m%dT%H%M%SZ",time.gmtime())+"_"+_stable_hash(cfg)
    return ExperimentRecord(
        experiment_id=eid,timestamp_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),strategy="walk_forward_logistic",target=TARGET,universe=list(ASSETS),
        data_mode=DATA_MODE,seed=SEED,feature_version=_stable_hash(list(features_with_info.columns)),strategy_version="V2.0",
        n_trials=1,n_folds=len(folds),mean_oos_sharpe=float(folds.oos_sharpe.mean()),median_oos_sharpe=float(folds.oos_sharpe.median()),
        mean_oos_auc=float(folds.oos_auc.mean()),mean_oos_brier=float(folds.oos_brier.mean()),worst_oos_dd=float(folds.oos_max_dd.min()),
        total_oos_trades=int(folds.oos_trades.sum()),bootstrap_lo=float(boot["lo"]),bootstrap_hi=float(boot["hi"]),cost_5bps_sharpe=cost5,
        delay_1bar_sharpe=d1,placebo_mean_sharpe=float(placebo.mean_sharpe.mean()),information_incremental_sharpe=info_delta,pass_fail=status)

record=create_experiment_record(folds,pred_df,cost_table,delay_table,placebo,ablation)
registry_path=ARTIFACT_DIR/"experiment_registry.jsonl"
with registry_path.open("a") as f: f.write(json.dumps(asdict(record),default=str)+"\n")
print(asdict(record))
print("registry:",registry_path)

{'experiment_id': '20260906T205940Z_b6c3b22718e1f4b8', 'timestamp_utc': '2026-09-06T20:59:40Z', 'strategy': 'walk_forward_logistic', 'target': 'SPY', 'universe': ['SPY', 'QQQ', 'AAPL', 'MSFT', 'JPM', 'GLD', 'TLT'], 'data_mode': 'synthetic', 'seed': 42, 'feature_version': '5b7e6b5c01187254', 'strategy_version': 'V2.0', 'n_trials': 1, 'n_folds': 7, 'mean_oos_sharpe': 0.20729040930388656, 'median_oos_sharpe': 0.40851661995722477, 'mean_oos_auc': 0.5018150355005083, 'mean_oos_brier': 0.2935824284459891, 'worst_oos_dd': -0.13923118375294274, 'total_oos_trades': 902, 'bootstrap_lo': -0.7901314478556348, 'bootstrap_hi': 0.7048613916766397, 'cost_5bps_sharpe': -0.01430583487225988, 'delay_1bar_sharpe': -0.031037413651382764, 'placebo_mean_sharpe': 0.37329908721022603, 'information_incremental_sharpe': 0.0, 'pass_fail': 'REVIEW'}
registry: /mnt/data/research_artifacts/experiment_registry.jsonl


## 13. Research decision gate

This is intentionally a **gate**, not a ranking by raw Sharpe. A `REVIEW` result means the strategy remains a research candidate and should not be promoted to paper/live execution.

In [16]:
def decision_gate(record, folds, cost_table, delay_table):
    criteria={
        "positive_median_oos_sharpe": bool(record.median_oos_sharpe>0),
        "survives_5bps": bool(record.cost_5bps_sharpe>0),
        "survives_1bar_delay": bool(record.delay_1bar_sharpe>0),
        "bootstrap_has_positive_median": bool(record.bootstrap_hi>record.bootstrap_lo and (record.bootstrap_lo>0)),
        "not_single_fold": bool(record.n_folds>=2 and (folds.oos_sharpe>0).sum()>=max(2,math.ceil(record.n_folds*0.5))),
        "dd_below_50pct": bool(abs(record.worst_oos_dd)<0.50),
    }
    passed=sum(criteria.values())
    status="PROMOTE_TO_PAPER_REVIEW" if passed>=5 else "RESEARCH_ONLY"
    return status, criteria

status,criteria=decision_gate(record,folds,cost_table,delay_table)
print("DECISION:",status)
print(json.dumps(criteria,indent=2))

DECISION: RESEARCH_ONLY
{
  "positive_median_oos_sharpe": true,
  "survives_5bps": false,
  "survives_1bar_delay": false,
  "bootstrap_has_positive_median": false,
  "not_single_fold": true,
  "dd_below_50pct": true
}


## 14. Export

The final artifacts are machine-readable so results can be compared across runs without copy/pasting notebook outputs.

In [17]:
outputs={
    "record":asdict(record),
    "decision":status,
    "criteria":criteria,
    "folds":folds.to_dict(orient="records"),
    "cost_sensitivity":cost_table.to_dict(orient="records"),
    "delay_sensitivity":delay_table.to_dict(orient="records"),
    "regime":regimes.to_dict(orient="records"),
    "ablation":ablation.to_dict(orient="records"),
    "placebo":placebo.to_dict(orient="records"),
    "parameter_perturbation":perturbations.to_dict(orient="records"),
    "portfolio_risk":portfolio_stats,
}
out_path=ARTIFACT_DIR/f"{record.experiment_id}_results.json"
out_path.write_text(json.dumps(outputs,indent=2,default=str))
folds.to_csv(ARTIFACT_DIR/f"{record.experiment_id}_folds.csv",index=False)
cost_table.to_csv(ARTIFACT_DIR/f"{record.experiment_id}_costs.csv",index=False)
print(out_path)

/mnt/data/research_artifacts/20260906T205940Z_b6c3b22718e1f4b8_results.json


## 15. Next-stage hooks

The engine is intentionally ready for the remaining planned layers:

1. **Alternative-data adapters:** SEC/FRED/news/Reddit/X can populate the event schema without changing the model contract.
2. **Triple-barrier labels:** replace `make_labels` with path-dependent labels while retaining the same split/evaluation interfaces.
3. **Controlled strategy discovery:** wrap `run_experiment` with bounded, seeded parameter search and record `n_trials` in the registry.
4. **Cross-asset validation:** run the same experiment function on independent universes and compare the same standardized metrics.
5. **Paper trading:** consume the exact production feature code and persist signals/fills/latency before any capital is exposed.

### Do not bypass the decision gate
A high raw backtest return is not sufficient for promotion. The objective remains a robust, reproducible, economically plausible OOS edge that survives costs, delays, regime changes, and execution effects.